In [1]:
import json
import pickle
from pathlib import Path

In [15]:
INPUT_JSON   = Path("../retrieval/VD/locisimiles-e5-xmlrob/results-locisimiles-e5-xmlrob.json")  
OUTPUT_JSON  = Path("../retrieval/VD/locisimiles-e5-xmlrob/retrieval_results_locisimiles-e5-xmlrob.json")
OUTPUT_PKL   = Path("../retrieval/VD/locisimiles-e5-xmlrob/retrieval_results_locisimiles-e5-xmlrob.pkl")

MODEL_NAME   = "locisimiles-e5-xmlrob"                     
INDEX_TYPE   = "LociSimiles pipeline"   
MAX_K        = 20
K_VALUES     = [5, 10, 20]

In [16]:
with open(INPUT_JSON, "r", encoding="utf-8") as f:
    raw = json.load(f)

print(f"Loaded {len(raw)} queries")
# Preview query
first_key = next(iter(raw))
print(f"\nExample query id : {first_key}")
print(f"Candidates       : {len(raw[first_key])}")
print("First candidate  :", raw[first_key][0])

Loaded 19466 queries

Example query id : 10015_sent_0
Candidates       : 20
First candidate  : {'source_id': '022_Hieronymus-Stridonensis_Epistolae_window_1994', 'source_text': 'veriora, ut absque ullo rancore stomachi, in Scripturarum disputatione versemur. Quo pacto enim possumus in hac disputatione sine rancore versari, si me laedere paras? aut si non paras, quomodo ego, te non laedente, abs te laesus juste expostularem 730 quod probare ante debuisses meum esse sermonem, et sic rescribere, hoc est et sic laedere? Nisi enim rescribendo laesisses, ego juste expostulare non possem. Proinde cum ita rescribis, ut laedas, quis locus nobis relinquitur in disputatione Scripturarum sine ullo rancore versandi? Ego quidem absit ut laedar, si mihi certa ratione volueris et potueris demonstrare illud ex epistola Apostoli, vel quid aliud Scripturarum sanctarum te verius intellexisse, quam me: imo vero absit, ut non cum gratiarum actione lucris meis deputem, SI FUERO TE docente instructus, aut eme

In [17]:
retrieval_results = {} 
total_candidates = 0

for query_id, candidates in raw.items():
    transformed = []
    for cand in candidates:
        transformed.append((cand["source_id"], cand["candidate_score"]))
        total_candidates += 1
    retrieval_results[query_id] = transformed

In [18]:
# JSON 
output_data = {
    "metadata": {
        "model_name": MODEL_NAME,
        "total_queries": len(retrieval_results),
        "total_candidates": total_candidates,
        "max_k": MAX_K,
        "k_values": K_VALUES,
        "index_type": INDEX_TYPE,
    },
    "retrieval_results": {
        query_id: [
            {
                "candidate_id": cand_id,
                "similarity": score,
                "rank": rank,
            }
            for rank, (cand_id, score) in enumerate(candidates, 1)
        ]
        for query_id, candidates in retrieval_results.items()
    }
}

In [19]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

# PKL — tuples only, same as FAISS
with open(OUTPUT_PKL, "wb") as f:
    pickle.dump(retrieval_results, f)

In [20]:
# save judgements for threshold analysis
OUTPUT_PKL_judge   = Path("../retrieval/VD/locisimiles-e5-xmlrob/retrieval_results_locisimiles_judgement-e5-xmlrob.pkl")

In [21]:
retrieval_results = {} 
total_candidates = 0

for query_id, candidates in raw.items():
    transformed = []
    for cand in candidates:
        transformed.append((cand["source_id"], cand["judgment_score"]))
        total_candidates += 1
    retrieval_results[query_id] = transformed

# PKL — tuples only
with open(OUTPUT_PKL_judge, "wb") as f:
    pickle.dump(retrieval_results, f)